# Stock options — IV and percentage premiums
## Bloomberg BQuant · arbitrary equities · puts/calls · choose your output columns

**Run All**, then choose your inputs in the dashboard and click **Load / rebuild**. No market-data request starts automatically. This notebook is self-contained; the separate Python file and earlier versions are not required.

**What changed in v3**

- **IV %** is the first/default ranking metric. Percentage premiums are available as **premium / stock last %** and **premium / strike %**. All three appear in the concise default table; raw premium, bid, ask and diagnostic fields are optional columns.
- **Custom stocks** and **Single stock** accept equities directly, without an S&P 500 membership check. Examples of input syntax: `SNDK UW Equity, SOFI US Equity, 700 HK Equity`. Bare symbols such as `SOFI` default to `SOFI US Equity`. Use commas or newlines to separate full IDs. A requested equity still needs an available Bloomberg option chain and the relevant data permissions.
- Expand **Columns to show / export** *before loading* to choose your columns. Ctrl/Cmd-click adds or removes columns. **Concise columns**, **All columns**, and **Clear columns** are shortcuts. Changes after loading apply locally. CSV uses the same columns.

**Defaults:** Custom stocks; calculated MID premium; puts; highest IV first; all listed strikes inside the selected range; 50 ranked rows. Choose **All S&P 500** only for an explicit index scan.

### IV and premium are different fields

`IV_PCT` is the option contract's Bloomberg BQL `ivol()` observation. It is **not** calculated from `PREMIUM_PCT_SPOT` or `PREMIUM_PCT_STRIKE`. The IV request uses field-default parameters and is independent of the premium quote selector; it is not labeled bid-IV, ask-IV, last-IV or mid-IV. IV does not change merely because you switch the premium basis.

`PREMIUM_PCT_SPOT = 100 × selected premium / stock last`  
`PREMIUM_PCT_STRIKE = 100 × selected premium / strike`

These are unannualized premium ratios, not investment returns. The HTML table prints `7.67%`; DataFrames and CSV store `7.67` as numeric percentage points. IV uses an explicit unit convention: **PERCENT** means `45 → 45%`; **DECIMAL** means `0.45 → 45%`. The default is PERCENT. No value-based unit guessing is performed. Change the unit under Advanced only after confirming the chosen field's convention in your BQuant environment.

**Your pasted example:** `131.4 / 1712.89 × 100 = 7.67%`; `131.4 / 1540 × 100 = 8.53%`. No IV was supplied in that pasted row, so none is invented here.


## 1 · Imports and field configuration

Use the Bloomberg-provided `bql` package with pandas, numpy, ipywidgets and IPython. Do not install the unrelated public PyPI package named `bql`. The IV adapter is configured here; field availability must be confirmed in your licensed session.

In [ ]:
# 1. Imports and configuration
from concurrent.futures import ThreadPoolExecutor
from datetime import timedelta
from pathlib import Path
import html
import math
import re
import warnings
import threading
import traceback

import numpy as np
import pandas as pd
try:
    import bql
    import ipywidgets as widgets
    from IPython.display import display, FileLink, clear_output
except ImportError as exc:
    raise ImportError(
        "Run this notebook in a Bloomberg BQuant Python environment with "
        "Bloomberg's bql and ipywidgets packages enabled. Do not install the "
        "unrelated public PyPI package named 'bql'."
    ) from exc

INDEX = "SPX Index"
DEFAULT_STOCK = "AAPL US Equity"
MAX_WORKERS = 1             # Serial by default. Optional: 2-4 independent workers.
QUOTE_BATCH_SIZE = 200      # Number of selected option contracts per quote request.
PAGE_SIZE = 100             # Display paging only; does not truncate the export.
AUTO_LOAD_SINGLE = False    # Choose stocks, ranges and columns BEFORE requesting data.
DEFAULT_EQUITY_MARKET = "US"  # Bare symbols such as SOFI become SOFI US Equity.

# Contract-specific Bloomberg BQL implied volatility; NOT historical/ATM stock vol.
# The native ivol() field is requested independently of LAST/BID/ASK/MID premiums.
IVOL_FIELD = "ivol"
IVOL_KWARGS = {}            # Only use parameters documented in your BQuant field help.
IVOL_INPUT_UNIT = "PERCENT" # 45 -> 45%. Choose DECIMAL in the UI for 0.45 -> 45%.
# No magnitude-based guessing: IV can legitimately exceed 100%. Check units against
# one known contract in your licensed BQuant field help if changing this field.

# Native BQL data items (not Excel/BDP OPT_* field mnemonics).
# If Bloomberg changes a field in your environment, this is the one place to edit.
FIELD_NAMES = {
    "EXPIRY": "expire_dt", "STRIKE": "strike_px", "SIDE": "put_call",
    "LAST": "px_last", "BID": "px_bid", "ASK": "px_ask",
    "NAME": "name", "STOCK_LAST": "px_last",
}


def utc_now():
    """Application retrieval time, NEVER represented as an exchange quote time."""
    return pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")


def market_date():
    """Use New York's calendar date rather than the notebook server's timezone."""
    return pd.Timestamp.now(tz="America/New_York").date()


def data_items(service, names):
    return {name: getattr(service.data, FIELD_NAMES[name])() for name in names}

## 2 · Bloomberg adapter and bounded chain selection

Custom inputs are queried directly. Only index mode requests S&P 500 membership. Failed or invalid stock inputs are isolated and reported. Stock/expiry/strike ranges bound the option-chain requests.

In [ ]:
# 2. Bloomberg query adapter and contract selection

def response_frame(response, aliases):
    """Join one current observation per ID; keep each item's DATE / CURRENCY.

    Joining by security ID is deliberate. Joining quotes and reference fields
    on DATE could lose contracts, and pivoting with mean could hide duplicates.
    Unexpected multi-row observations raise an error rather than silently choosing.
    """
    items = list(response)
    aliases = list(aliases)
    if len(items) != len(aliases):
        raise ValueError("BQL did not return one response item for every requested field.")
    parts = []
    for item, alias in zip(items, aliases):
        f = item.df().copy()
        if "ID" not in f.columns:
            f = f.reset_index()
        if "ID" not in f.columns:
            raise ValueError("BQL response has no ID column for " + alias)
        if alias not in f.columns:
            matches = [c for c in f.columns if str(c).upper() == alias.upper()]
            if len(matches) != 1:
                raise ValueError("Unexpected BQL columns for %s: %s" % (alias, list(f.columns)))
            f = f.rename(columns={matches[0]: alias})
        # Do not allow extra dimensions to be silently collapsed.
        if f["ID"].duplicated().any():
            raise ValueError("Multiple observations per ID for %s. Expected a snapshot, not a time series." % alias)
        kept = ["ID", alias]
        renames = {}
        for meta in ("DATE", "CURRENCY", "AS_OF_DATE", "UNIT", "UNITS"):
            if meta in f.columns:
                kept.append(meta)
                renames[meta] = alias + "_" + meta
        f = f[kept].rename(columns=renames)
        f["ID"] = f["ID"].astype(str)
        parts.append(f.set_index("ID"))
    if not parts:
        return pd.DataFrame(index=pd.Index([], name="ID"))
    return pd.concat(parts, axis=1, join="outer")


def execute_snapshot(service, universe, fields):
    # No historical date range, forward fill, or explicit cached-mode request.
    request = bql.Request(universe, fields)
    return response_frame(service.execute(request), fields.keys())


def load_members(service):
    f = execute_snapshot(service, service.univ.members(INDEX),
                         data_items(service, ["NAME", "STOCK_LAST"]))
    if f.empty:
        raise ValueError("No constituents returned for " + INDEX)
    f.index.name = "TICKER"
    f["NAME"] = f["NAME"].fillna("").astype(str)
    f["STOCK_LAST"] = pd.to_numeric(f["STOCK_LAST"], errors="coerce")
    f["STOCK_FETCHED_UTC"] = utc_now()
    return f.sort_index()


def normalize_stock_inputs(stocks, default_market=DEFAULT_EQUITY_MARKET):
    """Accept arbitrary equity tickers without testing index membership.

    Comma/semicolon/newline-separated full IDs are safest. Bare symbols default
    to the explicitly configured market (US). Full IDs retain their market.
    Bloomberg, not an S&P membership table, validates the resulting securities.
    """
    if stocks is None:
        raise ValueError("Enter stocks, or explicitly choose All S&P 500.")
    if isinstance(stocks, str):
        if re.search(r"\b(Index|Comdty|Curncy|Corp|Govt|Mtge|Muni)$", stocks.strip(), re.I):
            raise ValueError("Custom input accepts equities, not an index or another asset type.")
        parts = [x.strip() for x in re.split(r"[,;\n]+", stocks) if x.strip()]
        if len(parts) == 1 and not re.search(r"\bEquity$", parts[0], re.I):
            words = parts[0].split()
            # '700 HK' is one qualified ID. 'AAPL MSFT NVDA' is a bare-symbol list.
            if len(words) > 1 and not (len(words) == 2 and re.fullmatch(r"[A-Za-z]{2}", words[1])):
                parts = words
    else:
        parts = [str(x).strip() for x in stocks if str(x).strip()]
    if not parts:
        raise ValueError("Enter at least one ticker, e.g. SNDK UW Equity, SOFI US Equity.")
    market = str(default_market).strip().upper()
    if not re.fullmatch(r"[A-Z]{2}", market):
        raise ValueError("Default equity market must be a two-letter Bloomberg market code.")
    out = []
    for item in parts:
        words = item.split()
        if len(words) == 1:
            words += [market, "Equity"]
        elif len(words) == 2:
            words += ["Equity"]
        if (len(words) != 3 or words[-1].upper() != "EQUITY"
                or not re.fullmatch(r"[A-Za-z]{2}", words[1])
                or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9./_\-]*", words[0])):
            raise ValueError("Use a stock symbol or a full equity ID, separated by commas: " + item)
        identifier = words[0].upper() + " " + words[1].upper() + " Equity"
        if identifier not in out:
            out.append(identifier)
    return out


def load_custom_stocks(service, stocks):
    """Read entered securities directly; isolate invalid IDs and preserve input mapping.

    A batch can normalize exchange IDs. Unmatched IDs are queried individually
    instead of guessing which returned row belongs to an input. NAME failure
    must not suppress an available stock price.
    """
    requested = normalize_stock_inputs(stocks)
    errors, rows = [], []
    fields = data_items(service, ["NAME", "STOCK_LAST"])
    try:
        bulk = execute_snapshot(service, requested, fields)
    except Exception:
        bulk = pd.DataFrame()
    for ticker in requested:
        detail = ""
        if ticker in bulk.index:
            row = bulk.loc[ticker].to_dict()
            returned_id = ticker
        else:
            try:
                single = execute_snapshot(service, [ticker], fields)
            except Exception:
                parts = []
                for alias, expr in fields.items():
                    try:
                        part = execute_snapshot(service, [ticker], {alias: expr})
                        if len(part) > 1:
                            raise ValueError("More than one returned security for one stock input.")
                        parts.append(part)
                    except Exception as exc:
                        msg = "%s: %s" % (type(exc).__name__, exc)
                        errors.append({"STAGE": "Stocks", "SCOPE": ticker, "FIELD": alias, "ERROR": msg})
                        detail += alias + ": " + msg + "; "
                single = pd.concat(parts, axis=1) if parts else pd.DataFrame()
            if len(single) > 1:
                detail += "Ambiguous stock response; no input-to-ID mapping was guessed."
                row, returned_id = {}, ""
            elif len(single) == 1:
                row, returned_id = single.iloc[0].to_dict(), str(single.index[0])
            else:
                row, returned_id = {}, ""
                detail += "No stock observation returned; check ticker and entitlements."
        row["TICKER"] = ticker
        row["RETURNED_STOCK_ID"] = returned_id
        row["INPUT_DETAIL"] = detail
        row["STOCK_FETCHED_UTC"] = utc_now()
        rows.append(row)
    f = pd.DataFrame(rows).set_index("TICKER")
    if "NAME" not in f:
        f["NAME"] = ""
    f["NAME"] = f["NAME"].fillna("").astype(str)
    if "STOCK_LAST" not in f:
        f["STOCK_LAST"] = np.nan
    f["STOCK_LAST"] = pd.to_numeric(f["STOCK_LAST"], errors="coerce")
    return f, errors


def date_window(expiry_mode, exact, target_dte, window_days, today=None):
    today = today or market_date()
    target = today + timedelta(days=int(target_dte))
    if expiry_mode == "Exact date":
        if exact is None:
            raise ValueError("Select an exact expiration date.")
        if exact < today:
            raise ValueError("Expiration must not be earlier than today's New York date.")
        return exact, exact, exact
    return (max(today, target - timedelta(days=int(window_days))),
            target + timedelta(days=int(window_days)), target)


def select_contracts(frame, spot, start, end, target, mode, band_pct, strike_count,
                     strike_range=None, strike_mode="PCT_SPOT"):
    """Select actual listed expiries and strikes; no theoretical interpolation."""
    f = frame.copy()
    f["EXPIRY"] = pd.to_datetime(f["EXPIRY"], errors="coerce").dt.normalize()
    f["STRIKE"] = pd.to_numeric(f["STRIKE"], errors="coerce")
    f["SIDE"] = (f["SIDE"].astype(str).str.strip().str.upper()
                 .replace({"C": "CALL", "P": "PUT"}))
    structurally_valid = (f["EXPIRY"].notna() & np.isfinite(f["STRIKE"])
                          & f["STRIKE"].gt(0) & f["SIDE"].isin(["CALL", "PUT"]))
    bad_rows = int((~structurally_valid).sum())
    f = f.loc[structurally_valid].copy()
    bounds = strike_range if strike_range is not None else (100 - band_pct, 100 + band_pct)
    lower, upper = strike_bounds(spot, bounds, strike_mode)
    f = f[(f["EXPIRY"] >= pd.Timestamp(start)) & (f["EXPIRY"] <= pd.Timestamp(end))
          & f["STRIKE"].between(lower, upper)]
    if f.empty:
        return f, bad_rows
    if mode == "Nearest target DTE":
        expires = list(f["EXPIRY"].unique())
        chosen = min(expires, key=lambda d: (abs(pd.Timestamp(d) - pd.Timestamp(target)), pd.Timestamp(d)))
        f = f.loc[f["EXPIRY"] == chosen]
    if strike_count:  # 0 means all listed strikes inside the selected band.
        blocks = []
        for _, block in f.groupby("EXPIRY", sort=True):
            strikes = sorted(block["STRIKE"].unique(), key=lambda k: (abs(k - spot), k))
            blocks.append(block[block["STRIKE"].isin(strikes[:int(strike_count)])])
        f = pd.concat(blocks) if blocks else f.iloc[:0]
    return f.sort_values(["EXPIRY", "STRIKE", "SIDE"]), bad_rows


_thread_state = threading.local()


def discover_stock(ticker, member, config):
    """One bounded metadata request per stock, with a separate service per worker."""
    spot = pd.to_numeric(pd.Series([member.get("STOCK_LAST")]), errors="coerce").iloc[0]
    if not np.isfinite(spot) or spot <= 0:
        return pd.DataFrame(), "Missing/nonpositive stock price", str(member.get("INPUT_DETAIL", ""))
    try:
        if not hasattr(_thread_state, "service"):
            _thread_state.service = bql.Service()
        svc = _thread_state.service
        expiry = getattr(svc.data, FIELD_NAMES["EXPIRY"])()
        strike = getattr(svc.data, FIELD_NAMES["STRIKE"])()
        # These bounds are applied on Bloomberg's side, before downloading chains.
        condition = svc.func.and_(expiry >= config["start"].isoformat(),
                                  expiry <= config["end"].isoformat())
        lower, upper = strike_bounds(spot, config["strike_range"], config["strike_mode"])
        condition = condition.and_(strike >= lower)
        condition = condition.and_(strike <= upper)
        universe = svc.univ.filter(svc.univ.options(ticker), condition)
        f = execute_snapshot(svc, universe, data_items(svc, ["EXPIRY", "STRIKE", "SIDE"]))
        f, invalid = select_contracts(f, spot, config["start"], config["end"],
                                      config["target"], config["expiry_mode"],
                                      config.get("band", 20), config["strikes"],
                                      config["strike_range"], config["strike_mode"])
        if f.empty:
            return f, "No matching contracts", ("%d invalid metadata rows omitted" % invalid if invalid else "")
        f = f.reset_index()
        f["TICKER"] = ticker
        f["NAME"] = member.get("NAME", "")
        f["STOCK_LAST"] = spot
        f["STOCK_LAST_DATE"] = member.get("STOCK_LAST_DATE", pd.NaT)
        f["STOCK_CURRENCY"] = member.get("STOCK_LAST_CURRENCY", "")
        f["STOCK_FETCHED_UTC"] = member.get("STOCK_FETCHED_UTC", "")
        return f, "Selected", ("%d invalid metadata rows omitted" % invalid if invalid else "")
    except Exception as exc:
        return pd.DataFrame(), "Metadata error", "%s: %s" % (type(exc).__name__, exc)

## 3 · Quotes, independent IV retrieval and paired views

The IV request is separate from last/bid/ask retrieval. An unavailable IV field does not delete premiums. Missing IV remains blank; no historical-volatility or pricing-model fallback is used.

In [ ]:
# 3. Quote retrieval, quality checks, and call/put pairing

def fetch_quotes(service, metadata, progress=None):
    """Refresh only the selected contracts. Failed fields remain missing."""
    ids = metadata["ID"].drop_duplicates().tolist()
    blocks, errors = [], []
    for offset in range(0, len(ids), QUOTE_BATCH_SIZE):
        batch = ids[offset:offset + QUOTE_BATCH_SIZE]
        fields = data_items(service, ["LAST", "BID", "ASK"])
        try:
            quotes = execute_snapshot(service, batch, fields)
        except Exception as bulk_exc:
            # A missing/unauthorized bid field should not suppress an available last.
            parts = []
            for alias, expression in fields.items():
                try:
                    parts.append(execute_snapshot(service, batch, {alias: expression}))
                except Exception as exc:
                    errors.append({"STAGE": "Quotes", "SCOPE": "%d contracts, first: %s" % (len(batch), batch[0]),
                                   "FIELD": alias, "ERROR": "%s: %s" % (type(exc).__name__, exc)})
            quotes = pd.concat(parts, axis=1) if parts else pd.DataFrame(index=batch)
            if parts and not any(e["SCOPE"].endswith(batch[0]) for e in errors):
                errors.append({"STAGE": "Quotes", "SCOPE": batch[0], "FIELD": "Combined request",
                               "ERROR": "Recovered using separate fields: " + str(bulk_exc)})
        # IV failure never removes usable prices, and no historical-vol fallback is used.
        try:
            iv = execute_snapshot(service, batch,
                                  {"IV_RAW": getattr(service.data, IVOL_FIELD)(**IVOL_KWARGS)})
            quotes = pd.concat([quotes, iv], axis=1)
        except Exception as exc:
            errors.append({"STAGE": "IV", "SCOPE": "%d contracts, first: %s" % (len(batch), batch[0]),
                           "FIELD": IVOL_FIELD, "ERROR": "%s: %s" % (type(exc).__name__, exc)})
        quotes = quotes.reindex(batch)
        if "IV_RAW" not in quotes:
            quotes["IV_RAW"] = np.nan
        quotes["IV_SOURCE"] = "Bloomberg BQL " + IVOL_FIELD + "()"
        quotes["IV_BASIS"] = "Field default; independent of premium quote basis"
        quotes["IV_INPUT_UNIT"] = IVOL_INPUT_UNIT
        quotes.index.name = "ID"
        quotes["QUOTES_FETCHED_UTC"] = utc_now()
        blocks.append(quotes)
        if progress:
            progress(min(offset + len(batch), len(ids)), len(ids))
    quotes = pd.concat(blocks) if blocks else pd.DataFrame(index=pd.Index([], name="ID"))
    return metadata.merge(quotes.reset_index(), on="ID", how="left", validate="many_to_one"), errors


def enrich_quotes(raw, today=None, iv_unit=None):
    f = raw.copy()
    for col in ("LAST", "BID", "ASK"):
        if col not in f:
            f[col] = np.nan
        f[col] = pd.to_numeric(f[col], errors="coerce")
        for suffix in ("DATE", "CURRENCY"):
            key = col + "_" + suffix
            if key not in f:
                f[key] = pd.NaT if suffix == "DATE" else ""
    bid, ask = f["BID"], f["ASK"]
    bid_date = pd.to_datetime(f["BID_DATE"], errors="coerce", utc=True).dt.normalize()
    ask_date = pd.to_datetime(f["ASK_DATE"], errors="coerce", utc=True).dt.normalize()
    date_mismatch = bid_date.notna() & ask_date.notna() & bid_date.ne(ask_date)
    bc = f["BID_CURRENCY"].fillna("").astype(str)
    ac = f["ASK_CURRENCY"].fillna("").astype(str)
    currency_mismatch = bc.ne("") & ac.ne("") & bc.ne(ac)
    missing = ~np.isfinite(bid) | ~np.isfinite(ask)
    invalid = (bid < 0) | (ask <= 0) | (bid > ask)
    good = ~(missing | invalid | date_mismatch | currency_mismatch)
    f["MID"] = ((bid + ask) / 2).where(good)
    f["QUOTE_NOTE"] = np.select(
        [missing, bid.lt(0) | ask.le(0), bid.gt(ask), date_mismatch, currency_mismatch],
        ["Missing bid/ask", "Negative bid or nonpositive ask", "Crossed bid/ask",
         "Bid/ask observation dates differ", "Bid/ask currencies differ"],
        default="Indicative quote; intraday age unverified")
    f["DTE"] = (pd.to_datetime(f["EXPIRY"]) - pd.Timestamp(today or market_date())).dt.days
    spot = pd.to_numeric(f["STOCK_LAST"], errors="coerce")
    f["STRIKE_VS_SPOT_PCT"] = 100 * (f["STRIKE"] / spot.where(spot > 0) - 1)
    f = enrich_iv(f, iv_unit=iv_unit)
    return f


def enrich_iv(raw, iv_unit=None):
    """Convert contract IV to percentage points using an EXPLICIT unit convention.

    Never multiply small values heuristically; e.g. 0.5% is possible, as is 150%.
    The unit dropdown is local display/ranking configuration, not a new data query.
    """
    f = raw.copy()
    unit = str(iv_unit or IVOL_INPUT_UNIT).upper()
    if unit not in ("PERCENT", "DECIMAL"):
        raise ValueError("iv_unit must be PERCENT (45 means 45%) or DECIMAL (0.45 means 45%).")
    if "IV_RAW" not in f:
        # Also accept an already normalized DataFrame in the pure ranking function.
        f["IV_RAW"] = pd.to_numeric(f.get("IV_PCT", pd.Series(np.nan, index=f.index)), errors="coerce")
        if unit == "DECIMAL":
            f["IV_RAW"] = f["IV_RAW"] / 100.0
    value = pd.to_numeric(f["IV_RAW"], errors="coerce")
    f["IV_PCT"] = (value * (100.0 if unit == "DECIMAL" else 1.0)).where(np.isfinite(value) & value.gt(0))
    f["IV_INPUT_UNIT"] = unit
    f["IV_OBSERVATION_DATE"] = pd.to_datetime(
        f.get("IV_RAW_DATE", pd.Series(pd.NaT, index=f.index)), errors="coerce", utc=True)
    if "IV_SOURCE" not in f:
        f["IV_SOURCE"] = "Input DataFrame; verify source"
    if "IV_BASIS" not in f:
        f["IV_BASIS"] = "Field default; independent of premium quote basis"
    f["IV_NOTE"] = np.where(f["IV_PCT"].notna(),
        "Bloomberg/input IV; basis and intraday age not reconciled to selected premium",
        "IV unavailable or invalid; not estimated or replaced")
    return f


def pair_contracts(raw):
    """One row per stock/expiry/strike. Never average ambiguous option series.

    More than one contract on either side makes BOTH paired prices blank. All
    original contracts and quotes remain available in the individual-contract view.
    Deliverables are not verified; use full IDs to inspect nonstandard contracts.
    """
    if raw.empty:
        return pd.DataFrame()
    keys = ["TICKER", "EXPIRY", "STRIKE"]
    quote_cols = ["LAST", "BID", "ASK", "MID", "LAST_DATE", "BID_DATE", "ASK_DATE",
                  "LAST_CURRENCY", "BID_CURRENCY", "ASK_CURRENCY", "QUOTES_FETCHED_UTC", "QUOTE_NOTE",
                  "IV_RAW", "IV_PCT", "IV_OBSERVATION_DATE", "IV_SOURCE", "IV_BASIS", "IV_NOTE"]
    rows = []
    for (ticker, expiry, strike), block in raw.groupby(keys, sort=True):
        first = block.iloc[0]
        row = {"TICKER": ticker, "NAME": first["NAME"], "STOCK_LAST": first["STOCK_LAST"],
               "STOCK_LAST_DATE": first["STOCK_LAST_DATE"], "STOCK_CURRENCY": first["STOCK_CURRENCY"],
               "STOCK_FETCHED_UTC": first["STOCK_FETCHED_UTC"], "EXPIRY": expiry, "DTE": first["DTE"],
               "STRIKE": strike, "STRIKE_VS_SPOT_PCT": first["STRIKE_VS_SPOT_PCT"]}
        counts = block.groupby("SIDE").size()
        ambiguous = bool((counts > 1).any())
        row["PAIR_NOTE"] = ("Multiple series: inspect individual contracts" if ambiguous else
                            "Call or put absent" if len(counts) < 2 else "")
        for side in ("CALL", "PUT"):
            leg = block[block["SIDE"] == side]
            row[side + "_COUNT"] = len(leg)
            row[side + "_ID"] = " | ".join(leg["ID"].astype(str))
            for col in quote_cols:
                row[side + "_" + col] = (leg.iloc[0][col] if len(leg) == 1 and not ambiguous else
                                          np.nan if col in ("LAST", "BID", "ASK", "MID", "IV_RAW", "IV_PCT") else "")
        rows.append(row)
    return pd.DataFrame(rows)

## 4 · Local filtering and ranking

`rank_options(raw, ...)` only uses already-loaded data. IV ranking requires valid IV but can retain blank premiums. Percentage-premium rankings require a valid selected premium. No IV/premium substitution occurs.

In [ ]:
# 3b. Validation, percentage-premium calculations and IV/premium ranking

def checked_range(values, name, positive=False):
    """Validate a two-element inclusive numeric range."""
    if values is None:
        return None
    if isinstance(values, str) or len(values) != 2:
        raise ValueError(name + " must contain exactly (minimum, maximum).")
    try:
        lo, hi = float(values[0]), float(values[1])
    except (TypeError, ValueError) as exc:
        raise ValueError(name + " must contain two numbers.") from exc
    if not np.isfinite([lo, hi]).all() or lo > hi:
        raise ValueError(name + " must be finite, with minimum <= maximum.")
    if positive and lo <= 0:
        raise ValueError(name + " minimum must be greater than zero.")
    return lo, hi


def resolve_stock_selection(stocks, available):
    """Resolve exact Bloomberg IDs or unique short symbols. Never guess a ticker.

    String lists accept comma, semicolon or newline separators. A string of bare
    symbols separated by spaces also works. Full Bloomberg IDs must be separated
    by commas/newlines. Every requested stock must be in `available`.
    """
    ids = list(dict.fromkeys(str(x) for x in available))
    if stocks is None:
        return ids
    if isinstance(stocks, str):
        text = stocks.strip()
        if text.upper() in ("SPX", "SPX INDEX", "ALL S&P 500", "ALL"):
            return ids
        parts = [x.strip() for x in re.split(r"[,;\n]+", text) if x.strip()]
        if len(parts) == 1 and not re.search(r"\bEQUITY\b", text, re.I):
            parts = text.split()
    else:
        parts = [str(x).strip() for x in stocks if str(x).strip()]
    if not parts:
        raise ValueError("Enter at least one stock, for example AAPL, MSFT, NVDA.")
    full = {x.upper(): x for x in ids}
    short = {}
    for identifier in ids:
        short.setdefault(identifier.split()[0].upper(), []).append(identifier)
    selected, invalid = [], []
    for part in parts:
        key = part.upper()
        matches = [full[key]] if key in full else short.get(key, [])
        if len(matches) != 1:
            invalid.append(part + (" (ambiguous)" if len(matches) > 1 else " (not in this universe)"))
        elif matches[0] not in selected:
            selected.append(matches[0])
    if invalid:
        raise ValueError("Unresolved stocks: " + ", ".join(invalid) +
                         ". Use an exact Bloomberg ID from dashboard.members.index.")
    return selected


def strike_bounds(spot, strike_range, strike_mode="PCT_SPOT"):
    """ABSOLUTE uses native strike units; PCT_SPOT uses 100 * strike / stock last."""
    lo, hi = checked_range(strike_range, "Strike range", positive=True)
    mode = str(strike_mode).strip().upper()
    if mode == "ABSOLUTE":
        return lo, hi
    if mode != "PCT_SPOT":
        raise ValueError("strike_mode must be 'ABSOLUTE' or 'PCT_SPOT'.")
    if not np.isfinite(spot) or spot <= 0:
        raise ValueError("A positive finite stock price is required for relative strikes.")
    return spot * lo / 100.0, spot * hi / 100.0


def _currency_series(frame, name):
    if name not in frame:
        return pd.Series("", index=frame.index, dtype=object)
    # Preserve Bloomberg minor-unit currency codes; GBp must NOT become GBP.
    text = frame[name].fillna("").astype(str).str.strip()
    return text.map(lambda x: x if x in ("GBp", "ZAc") else x.upper()).replace(
        {"NAN": "", "NONE": "", "<NA>": ""})


def rank_options(raw, stocks=None, side="PUT", strike_range=None,
                 strike_mode="PCT_SPOT", stock_price_range=None,
                 expiry=None, dte_range=None, price_basis="MID",
                 rank_by="IV_PCT", top_n=50, one_per_stock=False,
                 positive_bid_only=False, max_spread_pct=None,
                 currency=None, allow_unknown_currency=True, today=None, iv_unit=None):
    """Filter ALREADY LOADED contracts and rank highest values first.

    Use dashboard.screen(...) to fetch a new universe/range from Bloomberg.
    This function makes no requests and cannot recover contracts not in `raw`.

    stocks: exact Bloomberg IDs or short symbols; None means all loaded stocks.
    side: PUT, CALL, or BOTH (P/C are accepted).
    strike_range: inclusive (low, high); (90, 100) with PCT_SPOT means K/S 90%-100%.
    stock_price_range: optional inclusive range of underlying last prices.
    expiry: exact date or ISO date string. dte_range: inclusive calendar days.
    price_basis: LAST, BID, ASK, MID. Missing/invalid prices are never substituted.
    rank_by: IV_PCT, PREMIUM_PCT_SPOT, PREMIUM_PCT_STRIKE, or legacy PREMIUM.
    top_n: number of rows returned; None or 0 returns all. Ties use stable ID keys.
    one_per_stock: retain the highest scoring contract per stock AND option side.
    positive_bid_only: additionally require a valid two-sided quote with bid > 0.
    max_spread_pct: optional ceiling for 100 * (ask-bid)/mid; requires valid mid.
    currency: optional quote-currency filter (None means all); no FX conversion.
    iv_unit: PERCENT or DECIMAL; None uses IVOL_INPUT_UNIT. No inferred rescaling.
    allow_unknown_currency: keep missing currency metadata, explicitly labeled.

    Premium ratios are NOT yields, implied volatility or model-based richness.
    Prices/strikes retain Bloomberg's native units. Contract multipliers and
    adjusted deliverables are not verified. Observation dates do not certify
    intraday freshness. A multi-request snapshot is not simultaneous.
    """
    if not isinstance(raw, pd.DataFrame):
        raise TypeError("raw must be a pandas DataFrame, normally dashboard.raw.")
    side = {"P": "PUT", "C": "CALL"}.get(str(side).upper(), str(side).upper())
    price_basis, rank_by = str(price_basis).upper(), str(rank_by).upper()
    strike_mode = str(strike_mode).upper()
    if side not in ("PUT", "CALL", "BOTH"):
        raise ValueError("side must be PUT, CALL or BOTH.")
    if price_basis not in ("LAST", "BID", "ASK", "MID"):
        raise ValueError("price_basis must be LAST, BID, ASK or MID.")
    rank_by = {"IV": "IV_PCT", "IMPLIED_VOL": "IV_PCT"}.get(rank_by, rank_by)
    if iv_unit is not None and str(iv_unit).upper() not in ("PERCENT", "DECIMAL"):
        raise ValueError("iv_unit must be PERCENT or DECIMAL.")
    if rank_by not in ("IV_PCT", "PREMIUM", "PREMIUM_PCT_SPOT", "PREMIUM_PCT_STRIKE"):
        raise ValueError("Unknown rank_by: " + rank_by)
    if strike_mode not in ("ABSOLUTE", "PCT_SPOT"):
        raise ValueError("strike_mode must be ABSOLUTE or PCT_SPOT.")
    strikes = checked_range(strike_range, "Strike range", positive=True)
    spots = checked_range(stock_price_range, "Stock-price range", positive=True)
    days = checked_range(dte_range, "DTE range")
    if days is not None and (days[0] < 0 or any(x != int(x) for x in days)):
        raise ValueError("DTE bounds must be nonnegative whole calendar days.")
    if top_n is not None and (isinstance(top_n, bool) or not np.isfinite(float(top_n))
                              or int(top_n) != float(top_n) or int(top_n) < 0):
        raise ValueError("top_n must be a nonnegative integer, or None.")
    if max_spread_pct is not None and (not np.isfinite(float(max_spread_pct))
                                       or float(max_spread_pct) < 0):
        raise ValueError("max_spread_pct must be finite and nonnegative, or None.")
    columns = ["RANK", "SIDE_RANK", "STOCK_SIDE_RANK", "TICKER", "NAME", "SIDE", "ID",
               "STOCK_LAST", "EXPIRY", "DTE", "STRIKE", "STRIKE_PCT_SPOT", "PRICE_BASIS",
               "IV_PCT", "PREMIUM", "PREMIUM_PCT_SPOT", "PREMIUM_PCT_STRIKE", "RANK_METRIC", "RANK_VALUE",
               "LAST", "BID", "ASK", "MID", "SPREAD_PCT", "PRICE_CURRENCY", "CURRENCY_CHECK",
               "PRICE_OBSERVATION_DATE", "QUOTE_NOTE", "IV_RAW", "IV_OBSERVATION_DATE", "IV_SOURCE", "IV_BASIS",
               "IV_INPUT_UNIT", "IV_NOTE", "QUOTES_FETCHED_UTC"]
    if raw.empty:
        return pd.DataFrame(columns=columns)
    required = {"TICKER", "ID", "SIDE", "STRIKE", "STOCK_LAST", "EXPIRY"}
    if not required.issubset(raw.columns):
        raise ValueError("Missing columns: " + ", ".join(sorted(required - set(raw.columns))))
    f = raw.copy()
    f["TICKER"] = f["TICKER"].astype(str)
    if stocks is not None:
        selected = resolve_stock_selection(stocks, f["TICKER"].unique())
        f = f[f["TICKER"].isin(selected)].copy()
    if f["ID"].duplicated().any():
        raise ValueError("Duplicate contract IDs in raw: expected one snapshot row per ID.")
    f["SIDE"] = f["SIDE"].astype(str).str.strip().str.upper().replace({"C": "CALL", "P": "PUT"})
    f["STRIKE"] = pd.to_numeric(f["STRIKE"], errors="coerce")
    f["STOCK_LAST"] = pd.to_numeric(f["STOCK_LAST"], errors="coerce")
    f["EXPIRY"] = pd.to_datetime(f["EXPIRY"], errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()
    valid = (f["EXPIRY"].notna() & np.isfinite(f["STRIKE"]) & f["STRIKE"].gt(0)
             & np.isfinite(f["STOCK_LAST"]) & f["STOCK_LAST"].gt(0)
             & f["SIDE"].isin(["CALL", "PUT"]))
    f = f.loc[valid].copy()
    f = enrich_quotes(f, today=today, iv_unit=iv_unit)
    f["STRIKE_PCT_SPOT"] = 100 * f["STRIKE"] / f["STOCK_LAST"]
    f = f.loc[f["DTE"] >= 0].copy()
    if side != "BOTH":
        f = f.loc[f["SIDE"] == side].copy()
    if spots is not None:
        f = f.loc[f["STOCK_LAST"].between(*spots)].copy()
    if strikes is not None:
        values = f["STRIKE"] if strike_mode == "ABSOLUTE" else f["STRIKE_PCT_SPOT"]
        # Compare strikes, not rounded display percentages, at inclusive boundaries.
        lower = strikes[0] if strike_mode == "ABSOLUTE" else f["STOCK_LAST"] * strikes[0] / 100
        upper = strikes[1] if strike_mode == "ABSOLUTE" else f["STOCK_LAST"] * strikes[1] / 100
        f = f.loc[(f["STRIKE"] >= lower) & (f["STRIKE"] <= upper)].copy()
    if expiry is not None:
        date = pd.Timestamp(expiry)
        if pd.isna(date):
            raise ValueError("expiry must be a valid date.")
        if date.tzinfo is not None:
            date = date.tz_localize(None)
        f = f.loc[f["EXPIRY"] == date.normalize()].copy()
    if days is not None:
        f = f.loc[f["DTE"].between(*days)].copy()
    f["PREMIUM"] = pd.to_numeric(f[price_basis], errors="coerce")
    f["PRICE_BASIS"] = price_basis
    f["SPREAD_PCT"] = (100 * (f["ASK"] - f["BID"]) / f["MID"]).where(f["MID"].gt(0))
    # An IV ranking may retain contracts with unavailable premiums, shown as blank.
    # Premium-based rankings still require a finite positive premium at the rank step.
    f["PREMIUM"] = f["PREMIUM"].where(np.isfinite(f["PREMIUM"]) & f["PREMIUM"].gt(0))
    if positive_bid_only:
        f = f.loc[f["BID"].gt(0) & f["MID"].notna()].copy()
    if max_spread_pct is not None:
        f = f.loc[f["SPREAD_PCT"].notna() & f["SPREAD_PCT"].le(float(max_spread_pct))].copy()
    if price_basis == "MID":
        bc, ac = _currency_series(f, "BID_CURRENCY"), _currency_series(f, "ASK_CURRENCY")
        pc = bc.where(bc.ne(""), ac)
        complete_price_currency = bc.ne("") & ac.ne("") & bc.eq(ac)
        bd = pd.to_datetime(f["BID_DATE"], errors="coerce", utc=True).dt.normalize()
        ad = pd.to_datetime(f["ASK_DATE"], errors="coerce", utc=True).dt.normalize()
        f["PRICE_OBSERVATION_DATE"] = bd.where(bd.notna() & ad.notna() & bd.eq(ad))
    else:
        pc = _currency_series(f, price_basis + "_CURRENCY")
        complete_price_currency = pc.ne("")
        f["PRICE_OBSERVATION_DATE"] = pd.to_datetime(f[price_basis + "_DATE"], errors="coerce", utc=True)
    sc = _currency_series(f, "STOCK_CURRENCY")
    kc = _currency_series(f, "STRIKE_CURRENCY")
    kc = kc.where(kc.ne(""), sc)
    spot_mismatch = pc.ne("") & sc.ne("") & pc.ne(sc)
    strike_mismatch = pc.ne("") & kc.ne("") & pc.ne(kc)
    f["PRICE_CURRENCY"] = pc.replace("", "UNKNOWN")
    f["CURRENCY_CHECK"] = np.where(complete_price_currency, "Quote currency available", "Quote currency metadata incomplete")
    f.loc[spot_mismatch | strike_mismatch, "CURRENCY_CHECK"] = "Currency/unit mismatch; affected percentage blank"
    f.loc[(sc.eq("") | kc.eq("")) & ~(spot_mismatch | strike_mismatch), "CURRENCY_CHECK"] = "Ratio unit metadata incomplete; verify"
    f["PREMIUM_PCT_SPOT"] = (100 * f["PREMIUM"] / f["STOCK_LAST"]).where(~spot_mismatch)
    f["PREMIUM_PCT_STRIKE"] = (100 * f["PREMIUM"] / f["STRIKE"]).where(~strike_mismatch)
    if not allow_unknown_currency:
        f.loc[~complete_price_currency | sc.eq(""), "PREMIUM_PCT_SPOT"] = np.nan
        f.loc[~complete_price_currency | kc.eq(""), "PREMIUM_PCT_STRIKE"] = np.nan
    needed = complete_price_currency.copy()
    if rank_by == "PREMIUM_PCT_SPOT":
        needed &= sc.ne("")
        f.loc[sc.eq(""), "CURRENCY_CHECK"] = "Stock currency metadata incomplete"
    elif rank_by == "PREMIUM_PCT_STRIKE":
        needed &= kc.ne("")
        f.loc[kc.eq(""), "CURRENCY_CHECK"] = "Strike/stock currency metadata incomplete"
    currency_mask = pd.Series(True, index=f.index)
    if currency is not None:
        wanted = str(currency).strip()
        wanted = wanted if wanted in ("GBp", "ZAc") else wanted.upper()
        if not wanted:
            raise ValueError("currency must be a currency code, or None.")
        currency_mask = pc.eq(wanted) | (pc.eq("") & allow_unknown_currency)
    if not allow_unknown_currency:
        currency_mask &= needed
    f = f.loc[currency_mask].copy()
    if rank_by == "PREMIUM" and currency is None:
        known = set(f["PRICE_CURRENCY"]) - {"UNKNOWN"}
        if len(known) > 1:
            raise ValueError("Raw premiums span multiple currencies; select one currency before ranking.")
    f["RANK_METRIC"] = rank_by
    f["RANK_VALUE"] = f[rank_by]
    f = f.loc[np.isfinite(f["RANK_VALUE"]) & f["RANK_VALUE"].gt(0)].copy()
    f = f.sort_values(["RANK_VALUE", "TICKER", "EXPIRY", "STRIKE", "SIDE", "ID"],
                      ascending=[False, True, True, True, True, True], kind="mergesort")
    if one_per_stock:
        f = f.drop_duplicates(["TICKER", "SIDE"], keep="first")
    f["RANK"] = np.arange(1, len(f) + 1)
    f["SIDE_RANK"] = f.groupby("SIDE", sort=False).cumcount() + 1
    f["STOCK_SIDE_RANK"] = f.groupby(["TICKER", "SIDE"], sort=False).cumcount() + 1
    eligible = len(f)
    if top_n:
        f = f.head(int(top_n)).copy()
    if "NAME" not in f:
        f["NAME"] = ""
    for column in columns:
        if column not in f:
            f[column] = np.nan
    extra = [c for c in f.columns if c not in columns]
    f = f[columns + extra].reset_index(drop=True)
    f.attrs["ranking_scope"] = "Loaded contracts only; no Bloomberg request was made by rank_options."
    f.attrs["eligible_rows_before_top_n"] = eligible
    f.attrs["price_units"] = "Native Bloomberg price units; no contract multiplier or FX conversion."
    return f

## 5 · Dashboard and column selection

The column selector is placed above the Load button and the result. It controls the visible table and CSV export without changing the ranking universe. Full results remain accessible through `dashboard.ranked` and `dashboard.raw`.

In [ ]:
def display_contract_metrics(raw, price_basis="MID", iv_unit=None):
    """Add metrics to EVERY loaded row; do not drop missing premiums or IV."""
    if raw.empty:
        return raw.copy()
    f = enrich_quotes(raw, iv_unit=iv_unit)
    f["PREMIUM"] = pd.to_numeric(f[price_basis], errors="coerce")
    f["PREMIUM"] = f["PREMIUM"].where(np.isfinite(f["PREMIUM"]) & f["PREMIUM"].gt(0))
    f["PRICE_BASIS"] = price_basis
    f["STRIKE_PCT_SPOT"] = 100 * f["STRIKE"] / f["STOCK_LAST"].where(f["STOCK_LAST"].gt(0))
    pc = _currency_series(f, price_basis + "_CURRENCY")
    if price_basis == "MID":
        pc = _currency_series(f, "BID_CURRENCY")
        pc = pc.where(pc.ne(""), _currency_series(f, "ASK_CURRENCY"))
    sc = _currency_series(f, "STOCK_CURRENCY")
    kc = _currency_series(f, "STRIKE_CURRENCY")
    kc = kc.where(kc.ne(""), sc)
    f["PREMIUM_PCT_SPOT"] = (100 * f["PREMIUM"] / f["STOCK_LAST"]).where(~(pc.ne("") & sc.ne("") & pc.ne(sc)))
    f["PREMIUM_PCT_STRIKE"] = (100 * f["PREMIUM"] / f["STRIKE"]).where(~(pc.ne("") & kc.ne("") & pc.ne(kc)))
    f["SPREAD_PCT"] = (100 * (f["ASK"] - f["BID"]) / f["MID"]).where(f["MID"].gt(0))
    return f


RANK_VIEW = "IV / premium ranking"
CONCISE_COLUMNS = ["RANK", "TICKER", "SIDE", "EXPIRY", "DTE", "STRIKE", "STRIKE_PCT_SPOT",
                   "IV_PCT", "PREMIUM_PCT_SPOT", "PREMIUM_PCT_STRIKE"]
COLUMN_LABELS = {
    "RANK": "Rank", "TICKER": "Ticker", "SIDE": "Side", "EXPIRY": "Expiry", "DTE": "DTE",
    "STRIKE": "Strike", "STOCK_LAST": "Stock last", "STRIKE_PCT_SPOT": "Strike / spot %",
    "IV_PCT": "IV %", "PREMIUM_PCT_SPOT": "Premium / spot %", "PREMIUM_PCT_STRIKE": "Premium / strike %",
    "PREMIUM": "Premium (raw)", "PRICE_BASIS": "Premium quote basis", "IV_RAW": "IV (raw)",
    "CALL_IV_PCT": "Call IV %", "PUT_IV_PCT": "Put IV %",
    "CALL_PREMIUM_PCT_SPOT": "Call premium / spot %", "PUT_PREMIUM_PCT_SPOT": "Put premium / spot %",
    "CALL_PREMIUM_PCT_STRIKE": "Call premium / strike %", "PUT_PREMIUM_PCT_STRIKE": "Put premium / strike %",
}
RANK_ALL_COLUMNS = list(dict.fromkeys(CONCISE_COLUMNS + [
    "NAME", "STOCK_LAST", "PRICE_BASIS", "PREMIUM", "LAST", "BID", "ASK", "MID", "SPREAD_PCT",
    "SIDE_RANK", "STOCK_SIDE_RANK", "RANK_METRIC", "RANK_VALUE", "ID", "IV_RAW", "IV_SOURCE", "IV_BASIS",
    "IV_INPUT_UNIT", "IV_OBSERVATION_DATE", "IV_NOTE", "PRICE_CURRENCY", "CURRENCY_CHECK",
    "PRICE_OBSERVATION_DATE", "QUOTES_FETCHED_UTC", "QUOTE_NOTE", "STOCK_LAST_DATE", "STOCK_CURRENCY",
    "STOCK_FETCHED_UTC", "LAST_DATE", "BID_DATE", "ASK_DATE", "LAST_CURRENCY", "BID_CURRENCY",
    "ASK_CURRENCY", "STRIKE_CURRENCY", "IV_RAW_DATE", "IV_RAW_AS_OF_DATE", "IV_RAW_UNIT", "IV_RAW_UNITS"]))
PAIR_CONCISE_COLUMNS = ["TICKER", "EXPIRY", "DTE", "STRIKE", "STRIKE_PCT_SPOT", "CALL_IV_PCT", "PUT_IV_PCT",
                       "CALL_PREMIUM_PCT_SPOT", "PUT_PREMIUM_PCT_SPOT", "CALL_PREMIUM_PCT_STRIKE", "PUT_PREMIUM_PCT_STRIKE"]
VIEW_DEFAULT_COLUMNS = {
    RANK_VIEW: CONCISE_COLUMNS,
    "Individual contracts": [c for c in CONCISE_COLUMNS if c != "RANK"],
    "Call / put pairs": PAIR_CONCISE_COLUMNS,
    "Stock coverage": ["TICKER", "NAME", "STATUS", "CONTRACTS", "CONTRACTS_WITH_IV", "DETAIL"],
    "Errors": ["STAGE", "SCOPE", "FIELD", "ERROR"],
}
VIEW_ALL_COLUMNS = {
    RANK_VIEW: RANK_ALL_COLUMNS,
    "Individual contracts": [c for c in RANK_ALL_COLUMNS if c not in ("RANK", "SIDE_RANK", "STOCK_SIDE_RANK", "RANK_METRIC", "RANK_VALUE")],
    "Call / put pairs": list(dict.fromkeys(PAIR_CONCISE_COLUMNS + ["NAME", "STOCK_LAST", "PRICE_BASIS", "PAIR_NOTE"] +
        [side + "_" + col for side in ("CALL", "PUT") for col in (
            "PREMIUM", "LAST", "BID", "ASK", "MID", "ID", "COUNT", "IV_RAW", "IV_SOURCE", "IV_BASIS",
            "IV_OBSERVATION_DATE", "IV_NOTE", "QUOTE_NOTE", "LAST_DATE", "BID_DATE", "ASK_DATE", "QUOTES_FETCHED_UTC")])) ,
    "Stock coverage": VIEW_DEFAULT_COLUMNS["Stock coverage"] + ["CONTRACTS_WITH_LAST", "CONTRACTS_WITH_VALID_MID"],
    "Errors": VIEW_DEFAULT_COLUMNS["Errors"],
}


def column_label(name):
    return COLUMN_LABELS.get(name, name.replace("_", " ").title())


def format_table_html(frame):
    """HTML display uses % signs; returned/exported data remain numeric percentage points."""
    shown = frame.copy()
    for col in shown:
        if pd.api.types.is_datetime64_any_dtype(shown[col]):
            shown[col] = shown[col].dt.strftime("%Y-%m-%d").fillna("")
        elif col.endswith("_PCT") or "_PCT_" in col:
            shown[col] = shown[col].map(lambda x: "—" if pd.isna(x) else "%.2f%%" % float(x))
    shown = shown.rename(columns=column_label)
    return shown.to_html(index=False, escape=True, na_rep="—", border=0,
                         classes="bq-options-table", float_format=lambda value: format(value, ",.2f"))


# 4. Notebook dashboard (ipywidgets only; no Dash/Streamlit server required)
class OptionsDashboard:
    def __init__(self):
        self.service = bql.Service()
        self.members = pd.DataFrame()
        self.metadata = pd.DataFrame()
        self.raw = pd.DataFrame()
        self.paired = pd.DataFrame()
        self.coverage = pd.DataFrame()
        self.errors = pd.DataFrame()
        self.loaded_config = None
        self.last_load_ok = False
        self.last_load_error = None
        self._suspend_updates = False
        self._busy = False
        self._rendering = False
        wide = widgets.Layout(width="310px")
        style = {"description_width": "initial"}
        self.mode = widgets.ToggleButtons(options=["Single stock", "Custom stocks", "All S&P 500"], description="Universe:", value="Custom stocks")
        self.stock = widgets.Combobox(options=[], value=DEFAULT_STOCK, ensure_option=False,
                                      description="Stock:", layout=wide, style=style)
        self.selected_stocks = widgets.Textarea(value="SNDK UW Equity, SOFI US Equity", description="Stocks:",
            placeholder="Any equities: SNDK UW Equity, SOFI US Equity, 700 HK Equity", style=style,
            disabled=True, layout=widgets.Layout(width="640px", height="65px"))
        self.stock_range_on = widgets.Checkbox(value=False, description="Filter stock last-price range", indent=False)
        self.stock_min = widgets.FloatText(value=50, description="Stock min:", style=style, layout=wide, disabled=True)
        self.stock_max = widgets.FloatText(value=500, description="Stock max:", style=style, layout=wide, disabled=True)
        self.expiry_mode = widgets.Dropdown(options=["Nearest target DTE", "Exact date", "All expiries in window", "DTE range"],
                                            description="Expiry selection:", layout=wide, style=style)
        self.target = widgets.BoundedIntText(value=30, min=0, max=1095, description="Target DTE:", style=style,
                                             layout=widgets.Layout(width="200px"))
        self.window = widgets.BoundedIntText(value=15, min=0, max=365, description="Window +/- days:", style=style,
                                             layout=widgets.Layout(width="210px"))
        self.exact = widgets.DatePicker(value=market_date() + timedelta(days=30), description="Exact expiry:",
                                        disabled=True, style=style, layout=wide)
        self.dte_min = widgets.BoundedIntText(value=20, min=0, max=3650, description="DTE min:", style=style,
                                              layout=widgets.Layout(width="200px"), disabled=True)
        self.dte_max = widgets.BoundedIntText(value=45, min=0, max=3650, description="DTE max:", style=style,
                                              layout=widgets.Layout(width="200px"), disabled=True)
        self.strike_mode = widgets.Dropdown(options=[("Strike as % of spot", "PCT_SPOT"), ("Absolute strike", "ABSOLUTE")],
                                            value="PCT_SPOT", description="Strike mode:", style=style, layout=wide)
        self.strike_min = widgets.FloatText(value=80, description="Strike min:", style=style, layout=widgets.Layout(width="200px"))
        self.strike_max = widgets.FloatText(value=120, description="Strike max:", style=style, layout=widgets.Layout(width="200px"))
        self.strikes = widgets.Dropdown(options=[("All within range", 0), ("5 nearest spot", 5), ("11 nearest spot", 11), ("21 nearest spot", 21)], value=0,
                                        description="Strikes / expiry:", style=style, layout=wide)
        self.load_btn = widgets.Button(description="Load / rebuild", button_style="primary", icon="refresh")
        self.quote_btn = widgets.Button(description="Refresh quotes only", disabled=True)
        self.progress = widgets.IntProgress(value=0, min=0, max=1, description="Progress:", layout=widgets.Layout(width="420px"))
        self.status = widgets.HTML("Click Load / rebuild to request Bloomberg data.")
        self.snapshot = widgets.HTML("")
        self.view = widgets.Dropdown(options=[RANK_VIEW, "Call / put pairs", "Individual contracts", "Stock coverage", "Errors"],
                                     description="Table:", style=style, layout=wide)
        self.price = widgets.Dropdown(options=[("Last", "LAST"), ("Mid (calculated)", "MID"), ("Bid", "BID"), ("Ask", "ASK")],
                                      value="MID", description="Premium basis:", style=style, layout=wide)
        self.rank_side = widgets.Dropdown(options=[("Puts", "PUT"), ("Calls", "CALL"), ("Both", "BOTH")],
                                           value="PUT", description="Rank side:", style=style, layout=wide)
        self.rank_metric = widgets.Dropdown(options=[("Implied volatility % (highest first)", "IV_PCT"),
            ("Premium / stock last %", "PREMIUM_PCT_SPOT"), ("Premium / strike %", "PREMIUM_PCT_STRIKE"),
            ("Raw premium (advanced)", "PREMIUM")],
            value="IV_PCT", description="Rank by:", style=style, layout=widgets.Layout(width="380px"))

        self.rank_topn = widgets.IntText(value=50, description="Top N (0=all):", style=style, layout=widgets.Layout(width="200px"))
        self.rank_one_per = widgets.Checkbox(value=False, description="Highest per stock / side only", indent=False)
        self.rank_positive_bid = widgets.Checkbox(value=False, description="Require positive bid + valid mid", indent=False)
        self.rank_spread_on = widgets.Checkbox(value=False, description="Limit bid/ask spread", indent=False)
        self.rank_spread_max = widgets.FloatText(value=25, description="Max spread / mid %:", style=style,
                                                layout=wide, disabled=True)
        self.rank_unknown_currency = widgets.Checkbox(value=True, description="Allow missing currency metadata", indent=False)
        self.rank_currency = widgets.Text(value="", description="Currency (blank=all):", continuous_update=False,
            style=style, layout=wide)
        self.iv_units = widgets.Dropdown(options=[("Percent points: 45 means 45%", "PERCENT"),
            ("Decimal: 0.45 means 45%", "DECIMAL")], value=IVOL_INPUT_UNIT,
            description="IV input units:", style=style, layout=widgets.Layout(width="400px"))
        self.rank_info = widgets.HTML("")
        self._columns_changing = False
        self._column_view = RANK_VIEW
        self._column_choices = {k: list(v) for k, v in VIEW_DEFAULT_COLUMNS.items()}
        self._column_available = {k: list(v) for k, v in VIEW_ALL_COLUMNS.items()}
        self.columns = widgets.SelectMultiple(options=[(column_label(c), c) for c in RANK_ALL_COLUMNS],
            value=tuple(CONCISE_COLUMNS), rows=9, description="Show columns:", style=style,
            layout=widgets.Layout(width="550px"))
        self.cols_compact = widgets.Button(description="Concise columns")
        self.cols_all = widgets.Button(description="All columns")
        self.cols_clear = widgets.Button(description="Clear columns")
        self.column_summary = widgets.HTML("10 columns selected. Choose columns before loading.")
        self.details = widgets.Checkbox(value=False, description="Show IDs, currencies and observation dates", indent=False,
                                        layout=widgets.Layout(width="370px"))
        self.search = widgets.Text(value="", description="Search:", placeholder="Ticker, company, contract, or error",
                                   continuous_update=False, style=style, layout=widgets.Layout(width="420px"))
        self.page = widgets.BoundedIntText(value=1, min=1, max=1, description="Page:", layout=widgets.Layout(width="145px"))
        self.prev_btn = widgets.Button(description="Previous", layout=widgets.Layout(width="90px"))
        self.next_btn = widgets.Button(description="Next", layout=widgets.Layout(width="75px"))
        self.page_info = widgets.HTML("")
        self.export_btn = widgets.Button(description="Export current table", disabled=True, icon="download")
        self.export_output = widgets.Output()
        self.table = widgets.HTML("")
        header = widgets.HTML("""
        <h2 style="margin-bottom:4px">Stock options · IV &amp; premium %</h2>
        <div>Any entered equity or S&amp;P 500 constituents · ranked puts and calls</div>
        <p><b>Choose stocks, ranges and output columns, then Load / rebuild.</b>
        Bloomberg IV is independent of the premium quote basis. Snapshots, not streaming.
        Percent premiums are ratios, not annualized returns. Quote age is unverified.</p>
        """)
        help_text = widgets.HTML("""
        <details><summary><b>Data and usage notes</b></summary>
        <p><b>Custom stocks:</b> enter comma/semicolon/newline-separated equity identifiers.
        Bare symbols use the US market; qualify non-US tickers, e.g. 700 HK Equity.
        No membership lookup is made in Custom stocks or Single stock mode. A valid equity
        may have no listed options, no matching expiry/strike, or unavailable data permissions.</p>
        <p><b>Strike ranges:</b> % of spot uses 100 × K/S; 90 to 100 means 90%–100% of stock last.
        Absolute uses native listed strike units. All within range avoids an ATM-only strike sample.
        Relative ranges assume compatible stock and strike units. Use absolute bounds and inspect
        unit metadata for markets whose strike and stock price conventions differ.</p>
        <p><b>Expiries:</b> nearest target may differ across stocks. Exact date uses a common expiry.
        DTE means calendar days from the New York date, not exchange-specific cutoff times.
        Same-day contracts may already have stopped trading.</p>
        <p><b>IV %:</b> contract-level BQL ivol() with field-default parameters; NOT historical
        volatility, stock ATM volatility, or IV inverted from the chosen premium. Field source,
        basis, raw value and observation date are selectable columns. PERCENT units means 45 → 45%;
        DECIMAL means 0.45 → 45%. No magnitude-based rescaling, model estimation, or fallback occurs.
        Check the field documentation in your licensed environment when changing IV settings.</p>
        <p><b>Premium %:</b> 100 × selected LAST/BID/ASK/MID divided by stock last or strike.
        No annualization, FX conversion or multiplier. IV is not either premium ratio. Premium ranks
        exclude missing/nonpositive premiums; IV ranks exclude missing/nonpositive IV but may retain
        missing premiums as blank. Positive-bid and spread filters are additional optional restrictions.</p>
        <p><b>Currency:</b> blank filter means all currencies. Known numerator/denominator mismatches
        blank the affected premium percentage. Missing metadata is allowed but unverified by default.
        Raw-premium ranking across multiple known currencies is blocked. Contract deliverables and
        nonstandard adjustments are not verified; inspect IDs before interpreting comparisons.</p>
        <p><b>Columns:</b> select the columns to display before or after loading (Ctrl/Cmd-click for multiple).
        Concise/All/Clear are shortcuts. Each view remembers its selection. CSV contains exactly the
        selected columns and all rows of the current searched Top N, not only the visible page.
        Numeric CSV/DataFrame percentages use percentage points: 7.67 means 7.67%, not 767%.</p>
        <p><b>Refreshing:</b> Load / rebuild fetches the chosen stocks, chains, quotes and IV.
        Refresh quotes only leaves the contract universe fixed, including expiry/strike selection;
        rebuild to discover new contracts. Ranking, unit display and columns update locally.
        IV field failures leave premiums intact; failed stocks/chains are reported in Coverage/Errors.
        An index-wide or batched scan is not a simultaneous market snapshot. Returned dates are
        observation dates, not certified exchange timestamps. Last/mid are not guaranteed executable.</p>
        <p>Rankings cover successfully loaded contracts only, with no fair-value verdict.
        Paired views leave ambiguous duplicate series blank rather than averaging. No trades are placed.
        Follow Bloomberg and exchange data-use rules when exporting.</p>
        </details>
        """)

        box = lambda children: widgets.HBox(children, layout=widgets.Layout(flex_flow="row wrap"))
        self.rank_controls = widgets.VBox([
            box([self.rank_side, self.rank_metric, self.rank_topn, self.rank_one_per]), self.rank_info,
        ])
        advanced = widgets.Accordion(children=[widgets.VBox([
            box([self.rank_positive_bid, self.rank_spread_on, self.rank_spread_max]),
            box([self.rank_currency, self.rank_unknown_currency]), box([self.iv_units]),
        ])])
        advanced.set_title(0, "Advanced: quote filters, currencies and IV units")
        advanced.selected_index = None
        columns_panel = widgets.Accordion(children=[widgets.VBox([
            widgets.HTML("Choose columns before loading. Ctrl/Cmd-click adds or removes individual columns."),
            self.columns, box([self.cols_compact, self.cols_all, self.cols_clear]),
        ])])
        columns_panel.set_title(0, "Columns to show / export")
        columns_panel.selected_index = None
        self.widget = widgets.VBox([
            header, self.mode, box([self.stock, self.selected_stocks]),
            box([self.stock_range_on, self.stock_min, self.stock_max]),
            box([self.expiry_mode, self.exact, self.dte_min, self.dte_max]),
            box([self.target, self.window]),
            box([self.strike_mode, self.strike_min, self.strike_max, self.strikes]),
            box([self.view, self.price]), self.rank_controls, advanced,
            self.column_summary, columns_panel,
            box([self.load_btn, self.quote_btn, self.progress]), self.status, self.snapshot,
            box([self.search, self.export_btn]),
            box([self.prev_btn, self.page, self.next_btn, self.page_info]),
            self.table, self.export_output, help_text,
        ])

        self._query_controls = [self.mode, self.stock, self.selected_stocks, self.stock_range_on,
            self.stock_min, self.stock_max, self.expiry_mode, self.target, self.window, self.exact,
            self.dte_min, self.dte_max, self.strike_mode, self.strike_min, self.strike_max, self.strikes]
        self._ranking_controls = [self.rank_side, self.rank_metric, self.rank_topn, self.rank_one_per,
            self.rank_positive_bid, self.rank_spread_on, self.rank_spread_max, self.rank_unknown_currency,
            self.rank_currency, self.iv_units]
        for control in self._query_controls:
            control.observe(self._filter_changed, names="value")
        for control in [self.view, self.price, self.details, self.search] + self._ranking_controls:
            control.observe(self._display_changed, names="value")
        self.page.observe(lambda change: self.render(), names="value")
        self.prev_btn.on_click(lambda _: self._turn_page(-1))
        self.next_btn.on_click(lambda _: self._turn_page(1))
        self.load_btn.on_click(self.load)
        self.quote_btn.on_click(self.refresh_quotes)
        self.export_btn.on_click(self.export)
        self.columns.observe(self._columns_changed, names="value")
        self.cols_compact.on_click(lambda _: self.set_columns(VIEW_DEFAULT_COLUMNS[self.view.value]))
        self.cols_all.on_click(lambda _: self.set_columns([v for _, v in self.columns.options]))
        self.cols_clear.on_click(lambda _: self.set_columns([]))
        self._sync_controls()

    def _message(self, text, error=False):
        self.status.value = ("<b>" + ("Error: " if error else "") + html.escape(str(text)) + "</b>")

    def _filter_changed(self, change=None):
        if self._busy or self._suspend_updates:
            return
        self._sync_controls()
        if self.loaded_config is not None:
            self._message("Query filters changed. Displayed prices/rankings use the PREVIOUS loaded selection. Click Load / rebuild to apply.")

    def _sync_controls(self):
        self.stock.disabled = self._busy or self.mode.value != "Single stock"
        self.selected_stocks.disabled = self._busy or self.mode.value != "Custom stocks"
        self.stock_min.disabled = self.stock_max.disabled = self._busy or not self.stock_range_on.value
        exact = self.expiry_mode.value == "Exact date"
        dte = self.expiry_mode.value == "DTE range"
        self.exact.disabled = self._busy or not exact
        self.target.disabled = self.window.disabled = self._busy or exact or dte
        self.dte_min.disabled = self.dte_max.disabled = self._busy or not dte
        self.rank_spread_max.disabled = self._busy or not self.rank_spread_on.value
        self.rank_controls.layout.display = "" if self.view.value == RANK_VIEW else "none"

    def _set_busy(self, value):
        self._busy = value
        for c in self._query_controls + self._ranking_controls:
            c.disabled = value
        self.load_btn.disabled = value
        self.quote_btn.disabled = value or self.metadata.empty
        self.export_btn.disabled = value or self.raw.empty
        self._sync_controls()

    def _display_changed(self, change=None):
        if self._rendering or self._suspend_updates:
            return
        self._sync_controls()
        self.page.value = 1
        self.render()

    def _turn_page(self, step):
        self.page.value = min(self.page.max, max(1, self.page.value + step))

    def _config(self):
        if self.expiry_mode.value == "DTE range":
            lo, hi = checked_range((self.dte_min.value, self.dte_max.value), "DTE range")
            today = market_date()
            start, end = today + timedelta(days=int(lo)), today + timedelta(days=int(hi))
            target = today + timedelta(days=int((lo + hi) // 2))
        else:
            start, end, target = date_window(self.expiry_mode.value, self.exact.value,
                                              self.target.value, self.window.value)
        bounds = checked_range((self.strike_min.value, self.strike_max.value), "Strike range", positive=True)
        spots = checked_range((self.stock_min.value, self.stock_max.value), "Stock-price range", positive=True) if self.stock_range_on.value else None
        return {"mode": self.mode.value, "expiry_mode": self.expiry_mode.value,
                "start": start, "end": end, "target": target,
                "strike_range": bounds, "strike_mode": self.strike_mode.value,
                "stock_price_range": spots, "strikes": int(self.strikes.value),
                "stock_list": self.selected_stocks.value}

    def _quote_progress(self, done, total):
        self.progress.max = max(total, 1)
        self.progress.value = done
        self._message("Retrieving selected option quotes: %s / %s contracts" % (done, total))

    def _finish_quotes(self, raw, errors):
        self.raw = enrich_quotes(raw, iv_unit=self.iv_units.value)
        self.paired = pair_contracts(self.raw)
        self.errors = pd.DataFrame(errors, columns=["STAGE", "SCOPE", "FIELD", "ERROR"])
        if not self.coverage.empty:
            for col in ["CONTRACTS", "CONTRACTS_WITH_LAST", "CONTRACTS_WITH_VALID_MID", "CONTRACTS_WITH_IV"]:
                self.coverage[col] = 0
            summary = self.raw.groupby("TICKER").agg(
                CONTRACTS=("ID", "size"), CONTRACTS_WITH_LAST=("LAST", "count"),
                CONTRACTS_WITH_VALID_MID=("MID", "count"), CONTRACTS_WITH_IV=("IV_PCT", "count"))
            for col in summary:
                self.coverage[col] = self.coverage["TICKER"].map(summary[col]).fillna(0).astype(int)
            no_prices = ((self.coverage["CONTRACTS"] > 0) &
                         self.coverage["TICKER"].map(self.raw.groupby("TICKER")[["LAST", "BID", "ASK"]]
                                                    .count().sum(axis=1)).fillna(0).eq(0))
            self.coverage.loc[no_prices, "STATUS"] = "Contracts found; prices unavailable"
            has_prices = (self.coverage["CONTRACTS"] > 0) & ~no_prices
            self.coverage.loc[has_prices, "STATUS"] = "Loaded (quote age unverified)"
            iv_only = no_prices & self.coverage["CONTRACTS_WITH_IV"].gt(0)
            self.coverage.loc[iv_only, "STATUS"] = "IV available; prices unavailable"
            aliases = self.coverage["STATUS"].eq("Selected") & self.coverage["CONTRACTS"].eq(0)
            self.coverage.loc[aliases, "STATUS"] = "Duplicate alias; contracts kept under first input"
        self.page.value = 1
        self.render()

    def _snapshot_label(self):
        c = self.loaded_config
        if c is None:
            return
        strikes = str(c["strikes"]) + " nearest" if c["strikes"] else "all in range"
        tickers = ", ".join(c["tickers"]) if len(c["tickers"]) <= 6 else "%s queried stocks" % len(c["tickers"])
        lo, hi = c["strike_range"]
        units = "% of stock last" if c["strike_mode"] == "PCT_SPOT" else "native strike units"
        self.snapshot.value = ("<p><b>Loaded selection:</b> %s | %s | %s to %s | strikes %g–%g %s | %s / expiry."
            "<br><b>Completed (UTC):</b> %s. This is application time, not quote time. %s stocks excluded by the stock-price range.</p>") % (
            html.escape(tickers), html.escape(c["expiry_mode"]), c["start"], c["end"],
            lo, hi, units, strikes, utc_now(), len(c.get("excluded_by_stock_range", [])))

    def load(self, _=None):
        if self._busy:
            return
        self._set_busy(True)
        self.last_load_ok = False
        self.last_load_error = None
        previous_exists = not self.raw.empty
        previous_state = (self.metadata, self.raw, self.paired, self.coverage, self.errors, self.loaded_config)
        try:
            config = self._config()
            input_errors = []
            if config["mode"] == "All S&P 500":
                self._message("Loading S&P 500 constituents and stock observations...")
                members = load_members(self.service)
                selected = members.index.tolist()
            else:
                entered = self.stock.value if config["mode"] == "Single stock" else config["stock_list"]
                selected = normalize_stock_inputs(entered)
                self._message("Loading entered equities directly (no index-membership restriction)...")
                members, input_errors = load_custom_stocks(self.service, selected)
            self.members = members
            self.stock.options = list(dict.fromkeys(list(self.stock.options) + members.index.tolist()))

            requested = list(selected)
            if config["stock_price_range"] is not None:
                lo, hi = config["stock_price_range"]
                selected = [t for t in selected if np.isfinite(members.loc[t, "STOCK_LAST"])
                            and lo <= members.loc[t, "STOCK_LAST"] <= hi]
            config["excluded_by_stock_range"] = [t for t in requested if t not in selected]
            config["tickers"] = selected
            self.progress.max = max(len(selected), 1)
            self.progress.value = 0
            self._message("Discovering options within the expiry and strike bounds...")
            blocks, errors = [], list(input_errors)
            coverage = [{"TICKER": t, "NAME": members.loc[t, "NAME"],
                         "STATUS": "Excluded by stock-price range", "DETAIL": "Outside range or stock price unavailable"}
                        for t in config["excluded_by_stock_range"]]
            workers = max(1, min(4, int(MAX_WORKERS)))
            # Bounded waves avoid queueing an entire index after a systemic access failure.
            wave_size = max(1, workers * 4)
            metadata_failures = 0
            stopped = False
            with ThreadPoolExecutor(max_workers=workers) as pool:
                for offset in range(0, len(selected), wave_size):
                    wave = selected[offset:offset + wave_size]
                    if workers == 1:
                        results = (discover_stock(ticker, members.loc[ticker].to_dict(), config) for ticker in wave)
                    else:
                        futures = [pool.submit(discover_stock, ticker, members.loc[ticker].to_dict(), config) for ticker in wave]
                        results = (future.result() for future in futures)
                    for ticker, result in zip(wave, results):
                        frame, status, error = result
                        coverage.append({"TICKER": ticker, "NAME": members.loc[ticker, "NAME"],
                                         "STATUS": status, "DETAIL": error})
                        if not frame.empty:
                            blocks.append(frame)
                        if error:
                            errors.append({"STAGE": "Metadata", "SCOPE": ticker, "FIELD": "Chain",
                                           "ERROR": error})
                        metadata_failures += int(status == "Metadata error")
                        self.progress.value += 1
                        self._message("Option-chain discovery: %s / %s stocks" % (self.progress.value, len(selected)))
                    if self.progress.value >= 8 and metadata_failures == self.progress.value:
                        for ticker in selected[offset + len(wave):]:
                            coverage.append({"TICKER": ticker, "NAME": members.loc[ticker, "NAME"],
                                             "STATUS": "Not attempted: repeated metadata failures", "DETAIL": "Resolve the errors, then rebuild."})
                        stopped = True
                        break
            metadata = pd.concat(blocks, ignore_index=True) if blocks else pd.DataFrame()
            if not metadata.empty and metadata["ID"].duplicated().any():
                count = int(metadata["ID"].duplicated().sum())
                metadata = metadata.drop_duplicates("ID", keep="first").copy()
                errors.append({"STAGE": "Metadata", "SCOPE": "Universe", "FIELD": "ID",
                    "ERROR": "%d duplicate contract IDs from stock aliases omitted; first input retained." % count})
            self.coverage = pd.DataFrame(coverage)
            self.loaded_config = config
            self.metadata = metadata
            if metadata.empty:
                self.raw, self.paired = pd.DataFrame(), pd.DataFrame()
                self.errors = pd.DataFrame(errors, columns=["STAGE", "SCOPE", "FIELD", "ERROR"])
                self.view.value = "Stock coverage"
                self.render()
                self._snapshot_label()
                self.last_load_ok = True
                self._message("No contracts selected. Review coverage/errors, stock prices and expiry/strike bounds.")
                return
            raw, quote_errors = fetch_quotes(self.service, metadata, self._quote_progress)
            self._finish_quotes(raw, errors + quote_errors)
            if self.view.value in ("Errors", "Stock coverage") and not errors:
                self.view.value = RANK_VIEW
            self._snapshot_label()
            self.last_load_ok = True
            represented = self.raw["TICKER"].nunique()
            self._message("Loaded %s contracts for %s / %s requested stocks. %s diagnostic messages.%s" % (
                len(self.raw), represented, len(selected), len(self.errors),
                " Scan stopped after repeated failures; see coverage." if stopped else ""))
        except Exception as exc:
            self.last_load_error = "%s: %s" % (type(exc).__name__, exc)
            self.metadata, self.raw, self.paired, self.coverage, self.errors, self.loaded_config = previous_state
            self.render()
            # Do not label retained data with a new successful snapshot time.
            self._message("%s: %s%s" % (type(exc).__name__, exc,
                          " Previous displayed prices have not been refreshed." if previous_exists else ""), error=True)
            traceback.print_exc()
        finally:
            self._set_busy(False)

    def refresh_quotes(self, _=None):
        if self._busy or self.metadata.empty:
            return
        self._set_busy(True)
        previous_state = (self.metadata, self.raw, self.paired, self.coverage.copy(), self.errors)
        try:
            self._message("Refreshing stock observations for the loaded contract selection...")
            meta = self.metadata.copy()
            tickers = meta["TICKER"].unique().tolist()
            stock_quotes = execute_snapshot(self.service, tickers, data_items(self.service, ["STOCK_LAST"]))
            stock_quotes["STOCK_FETCHED_UTC"] = utc_now()
            meta["STOCK_LAST"] = pd.to_numeric(meta["TICKER"].map(stock_quotes["STOCK_LAST"]), errors="coerce")
            meta["STOCK_LAST_DATE"] = (meta["TICKER"].map(stock_quotes["STOCK_LAST_DATE"])
                                       if "STOCK_LAST_DATE" in stock_quotes else pd.NaT)
            meta["STOCK_FETCHED_UTC"] = meta["TICKER"].map(stock_quotes["STOCK_FETCHED_UTC"])
            meta["STOCK_CURRENCY"] = (meta["TICKER"].map(stock_quotes["STOCK_LAST_CURRENCY"])
                                      if "STOCK_LAST_CURRENCY" in stock_quotes else "")
            raw, errors = fetch_quotes(self.service, meta, self._quote_progress)
            existing = self.errors[self.errors["STAGE"].isin(["Metadata", "Stocks"])].to_dict("records") if not self.errors.empty else []
            self.metadata = meta
            self._finish_quotes(raw, existing + errors)
            self._snapshot_label()
            self._message("Refreshed the loaded contract universe. Ranking bounds rechecked against updated stock last; rebuild to discover newly qualifying contracts. %s diagnostics." % len(self.errors))
        except Exception as exc:
            self.metadata, self.raw, self.paired, self.coverage, self.errors = previous_state
            self.render()
            self._message("Refresh failed; previous displayed prices retained. %s: %s" % (type(exc).__name__, exc), error=True)
            traceback.print_exc()
        finally:
            self._set_busy(False)

    @property
    def ranked(self):
        """Full Top-N ranked DataFrame, with metadata; reflects the LOADED query bounds."""
        c = self.loaded_config or {}
        return rank_options(self.raw, side=self.rank_side.value,
            strike_range=c.get("strike_range"), strike_mode=c.get("strike_mode", "PCT_SPOT"),
            stock_price_range=c.get("stock_price_range"), price_basis=self.price.value,
            rank_by=self.rank_metric.value, top_n=self.rank_topn.value,
            one_per_stock=self.rank_one_per.value, positive_bid_only=self.rank_positive_bid.value,
            max_spread_pct=self.rank_spread_max.value if self.rank_spread_on.value else None,
            currency=self.rank_currency.value.strip() or None,
            allow_unknown_currency=self.rank_unknown_currency.value, iv_unit=self.iv_units.value)

    def screen(self, stocks=None, strike_range=(90, 110), strike_mode="PCT_SPOT",
               side="PUT", price_basis="MID", rank_by="IV_PCT", expiry=None,
               dte_range=(20, 45), stock_price_range=None, top_n=50,
               one_per_stock=False, positive_bid_only=False, max_spread_pct=None,
               allow_unknown_currency=True, currency=None, iv_unit=None, columns=None):
        """Load Bloomberg data and return the selected output columns (numeric percentages).

        stocks=None or 'SPX' explicitly scans all returned S&P 500 constituents.
        Otherwise pass ANY supported equity ticker list, including non-members.
        columns optionally chooses the output columns before loading. dashboard.ranked
        retains all columns. All listed strikes inside the supplied range
        are requested, not an ATM-strike sample. expiry overrides dte_range.
        Use dashboard.coverage and dashboard.errors for partial retrievals.
        """
        if self._busy:
            raise RuntimeError("A request is already running in this dashboard.")
        # Validate ranking inputs before changing controls or making a request.
        rank_options(pd.DataFrame(), side=side, strike_range=strike_range, strike_mode=strike_mode,
                     stock_price_range=stock_price_range, dte_range=dte_range if expiry is None else None,
                     price_basis=price_basis, rank_by=rank_by, top_n=top_n, max_spread_pct=max_spread_pct,
                     currency=currency, iv_unit=iv_unit)
        bounds = checked_range(strike_range, "Strike range", positive=True)
        if bounds is None:
            raise ValueError("screen requires an explicit strike_range to bound Bloomberg requests.")
        days = checked_range(dte_range, "DTE range") if expiry is None else None
        if expiry is None and days is None:
            raise ValueError("Provide expiry or an explicit dte_range.")
        if days is not None and days[1] > 3650:
            raise ValueError("DTE maximum must be <= 3650 in this dashboard.")
        expiration = pd.Timestamp(expiry).date() if expiry is not None else None
        if expiration is not None and (pd.isna(expiration) or expiration < market_date()):
            raise ValueError("expiry must be a valid date on or after the New York calendar date.")
        self._suspend_updates = True
        try:
            is_all = stocks is None or (isinstance(stocks, str) and stocks.strip().upper() in ("SPX", "SPX INDEX", "ALL", "ALL S&P 500"))
            self.mode.value = "All S&P 500" if is_all else "Custom stocks"
            if not is_all:
                self.selected_stocks.value = stocks if isinstance(stocks, str) else ", ".join(str(x) for x in stocks)
            self.strike_mode.value = str(strike_mode).upper()
            self.strike_min.value, self.strike_max.value = bounds
            self.strikes.value = 0
            self.stock_range_on.value = stock_price_range is not None
            if stock_price_range is not None:
                self.stock_min.value, self.stock_max.value = checked_range(stock_price_range, "Stock-price range", positive=True)
            if expiration is not None:
                self.expiry_mode.value = "Exact date"
                self.exact.value = expiration
            else:
                self.expiry_mode.value = "DTE range"
                self.dte_min.value, self.dte_max.value = (int(days[0]), int(days[1]))
            self.rank_side.value = {"P": "PUT", "C": "CALL"}.get(str(side).upper(), str(side).upper())
            self.price.value = str(price_basis).upper()
            self.rank_metric.value = {"IV": "IV_PCT", "IMPLIED_VOL": "IV_PCT"}.get(str(rank_by).upper(), str(rank_by).upper())
            self.rank_topn.value = 0 if top_n is None else int(top_n)
            self.rank_one_per.value = bool(one_per_stock)
            self.rank_positive_bid.value = bool(positive_bid_only)
            self.rank_spread_on.value = max_spread_pct is not None
            if max_spread_pct is not None:
                self.rank_spread_max.value = float(max_spread_pct)
            self.rank_unknown_currency.value = bool(allow_unknown_currency)
            self.view.value = RANK_VIEW
            self.rank_currency.value = "" if currency is None else str(currency).strip()
            if iv_unit is not None:
                self.iv_units.value = str(iv_unit).upper()
            self._sync_columns()
            if columns is not None:
                self.set_columns(columns)
            self.search.value = ""
        finally:
            self._suspend_updates = False
            self._sync_controls()
        self.load()
        if not self.last_load_ok:
            raise RuntimeError(self.last_load_error or "Bloomberg request failed; see dashboard status.")
        if not self.errors.empty:
            warnings.warn("Some requests or data checks reported diagnostics. The ranking is among successfully loaded contracts only; inspect dashboard.coverage and dashboard.errors.", RuntimeWarning)
        self.view.value = RANK_VIEW
        self.render()
        return self.current_table()

    def _sync_columns(self, extra=()):
        if self._columns_changing:
            return
        self._columns_changing = True
        try:
            view = self.view.value
            if self._column_view != view:
                self._column_choices[self._column_view] = list(self.columns.value)
                self._column_view = view
            choices = list(dict.fromkeys(self._column_available[view] + list(extra)))
            self._column_available[view] = choices
            desired = self._column_choices.get(view, VIEW_DEFAULT_COLUMNS[view])
            self.columns.options = [(column_label(c), c) for c in choices]
            self.columns.value = tuple(c for c in desired if c in choices)
            self.column_summary.value = "<b>%d columns selected</b> · change under Columns to show / export." % len(self.columns.value)
        finally:
            self._columns_changing = False

    def _columns_changed(self, change=None):
        if self._columns_changing:
            return
        self._column_choices[self._column_view] = list(self.columns.value)
        self.column_summary.value = "<b>%d columns selected</b> · table and CSV use this selection." % len(self.columns.value)
        if not self._suspend_updates:
            self.render()

    def set_columns(self, columns):
        """Choose exactly these columns and their order; no Bloomberg request."""
        if isinstance(columns, str):
            columns = [x.strip() for x in columns.split(",") if x.strip()]
        columns = list(dict.fromkeys(columns))
        self._sync_columns()
        available = {v for _, v in self.columns.options}
        unknown = [c for c in columns if c not in available]
        if unknown:
            raise ValueError("Unknown columns for %s: %s" % (self.view.value, ", ".join(unknown)))
        self._column_choices[self.view.value] = columns
        self.columns.value = tuple(columns)
        return self

    def _view_frame(self):
        if self.view.value == RANK_VIEW:
            return self.ranked
        if self.view.value == "Stock coverage":
            return self.coverage.copy()
        if self.view.value == "Errors":
            return self.errors.copy()
        if self.view.value == "Individual contracts":
            return display_contract_metrics(self.raw, self.price.value, self.iv_units.value)
        f = pair_contracts(enrich_quotes(self.raw, iv_unit=self.iv_units.value)) if not self.raw.empty else pd.DataFrame()
        if f.empty:
            return f
        f["PRICE_BASIS"] = self.price.value
        f["STRIKE_PCT_SPOT"] = 100 * f["STRIKE"] / f["STOCK_LAST"].where(f["STOCK_LAST"].gt(0))
        metrics = display_contract_metrics(self.raw, self.price.value, self.iv_units.value)
        # Map actual single-contract IDs to ratios already checked for currency mismatch.
        for side in ("CALL", "PUT"):
            single = metrics.loc[metrics["SIDE"].eq(side)].set_index("ID")
            for col in ("PREMIUM", "PREMIUM_PCT_SPOT", "PREMIUM_PCT_STRIKE"):
                f[side + "_" + col] = f[side + "_ID"].map(single[col])
                f.loc[f["PAIR_NOTE"].str.startswith("Multiple series"), side + "_" + col] = np.nan
        return f

    def current_table(self, filtered=True, selected_columns=True):
        """Current view. Hidden fields remain searchable; output selection does not affect ranks."""
        f = self._view_frame()
        self._sync_columns(f.columns)
        if filtered and not f.empty and self.search.value.strip():
            mask = pd.Series(False, index=f.index)
            for col in f.select_dtypes(include=["object", "string"]).columns:
                mask |= f[col].fillna("").astype(str).str.contains(self.search.value.strip(), case=False, regex=False)
            f = f.loc[mask]
        if selected_columns:
            # Columns selected before load are preserved, including all-missing IV.
            f = f.reindex(columns=list(self.columns.value))
        return f


    def render(self):
        if self._rendering:
            return
        self._rendering = True
        try:
            f = self.current_table()
            n = len(f)
            iv_count = int(pd.to_numeric(self.raw.get("IV_RAW", pd.Series(dtype=float)), errors="coerce").gt(0).sum())
            self.rank_info.value = ("<div><b>Highest first:</b> %s · %s · premium basis %s. "
                "IV source: %s() (field default). %s / %s loaded contracts have positive IV. "
                "Missing IV is never replaced with premium %%.</div>" % (
                    html.escape(self.rank_side.label), html.escape(self.rank_metric.label), html.escape(self.price.label),
                    html.escape(IVOL_FIELD), iv_count, len(self.raw)))

            pages = max(1, math.ceil(n / PAGE_SIZE))
            self.page.max = pages
            self.page.value = min(self.page.value, pages)
            self.prev_btn.disabled = self.page.value <= 1
            self.next_btn.disabled = self.page.value >= pages
            start = (self.page.value - 1) * PAGE_SIZE
            self.page_info.value = "%s rows · page %s of %s · price selector: %s" % (n, self.page.value, pages, self.price.label)
            if len(f.columns) == 0:
                self.table.value = "<p>Select at least one column under Columns to show / export.</p>"
                return
            if f.empty:
                self.table.value = "<p>No eligible rows in this view. Check the selected side, ranges and quote filters; review Stock coverage / Errors, or clear Search.</p>"
                return
            shown = f.iloc[start:start + PAGE_SIZE].copy()
            table = format_table_html(shown)

            self.table.value = """<style>
            .bq-options-wrap {overflow:auto;max-height:620px;border:1px solid #8884;}
            .bq-options-table {border-collapse:collapse;font:12px sans-serif;white-space:nowrap;width:100%;}
            .bq-options-table th {position:sticky;top:0;background:var(--jp-layout-color2,#eee);padding:9px;text-align:right;}
            .bq-options-table td {padding:7px 10px;border-bottom:1px solid #8883;text-align:right;}
            </style><div class="bq-options-wrap">""" + table + "</div>"
        except Exception as exc:
            self.table.value = "<p><b>Ranking/display error:</b> " + html.escape(str(exc)) + "</p>"
            self.page_info.value = "No ranked result: fix the displayed filter error."
        finally:
            self._rendering = False

    def export(self, _=None):
        try:
            table = self.current_table()
            if table.empty:
                self._message("There are no matching rows to export.")
                return
            folder = Path("bquant_exports")
            folder.mkdir(exist_ok=True)
            stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
            prefix = "stock_options_iv_ranking_" if self.view.value == RANK_VIEW else "stock_options_"
            path = folder / (prefix + stamp + ".csv")
            table.to_csv(path, index=False)
            with self.export_output:
                clear_output(wait=True)
                print("Saved %s rows. Bloomberg/exchange data-use permissions still apply." % len(table))
                display(FileLink(str(path)))
        except Exception as exc:
            self._message("Export failed: " + str(exc), error=True)

## 6 · Launch

Run this cell, enter custom equities and range settings, expand Columns to show / export, then click Load / rebuild. `AUTO_LOAD_SINGLE=False` deliberately prevents any initial market-data request.

In [ ]:
# 5. Launch. Choose stocks, ranges and columns before any request.
dashboard = OptionsDashboard()
display(dashboard.widget)
if AUTO_LOAD_SINGLE:
    dashboard.load()

# dashboard.ranked: current full Top-N ranking, including metadata
# dashboard.raw: all loaded option contracts and quotes, before ranking filters
# dashboard.paired: side-by-side call/put quotes
# dashboard.coverage / dashboard.errors: completeness and request diagnostics
# dashboard.members: most recently requested stock-universe observations (custom or index)
# dashboard.screen(..., columns=[...]): query Bloomberg and return selected columns
# dashboard.set_columns([...]): change visible/exported columns without a query
# rank_options(dashboard.raw, ...): rerank only loaded contracts, without requests

## 7 · Optional function interface

The next cell does nothing unless `RUN_SCREEN_EXAMPLE=True`. `columns` chooses the returned columns **before** querying Bloomberg. `screen` updates the displayed dashboard as well. The full ranked DataFrame is always available as `dashboard.ranked`.

Use `side="CALL"` or `side="BOTH"` for calls. Change `rank_by` to `"PREMIUM_PCT_SPOT"` or `"PREMIUM_PCT_STRIKE"` to rank by a percentage premium. `stocks=None` explicitly scans all returned S&P 500 constituents; it is not the meaning of an empty custom-input box.


In [ ]:
RUN_SCREEN_EXAMPLE = False

if RUN_SCREEN_EXAMPLE:
    result = dashboard.screen(
        stocks=["SNDK UW Equity", "SOFI US Equity", "ASTS US Equity"],
        strike_mode="PCT_SPOT",
        strike_range=(90, 100),       # Listed strikes from 90% to 100% of stock last
        dte_range=(30, 90),           # Calendar days; use expiry="YYYY-MM-DD" for one date
        side="PUT",                  # PUT, CALL or BOTH
        price_basis="MID",           # Premium quote basis only; IV is independent
        rank_by="IV_PCT",            # Or PREMIUM_PCT_SPOT / PREMIUM_PCT_STRIKE
        top_n=50,
        columns=[
            "RANK", "TICKER", "SIDE", "EXPIRY", "STRIKE", "IV_PCT",
            "PREMIUM_PCT_SPOT", "PREMIUM_PCT_STRIKE",
        ],
    )
    # Result stays numeric; the dashboard above already shows percent signs.
    display(result)


## 8 · Change columns or rerank without new data requests

The next optional cell makes **no Bloomberg requests**. It cannot include new stocks, expiries or strikes that have not been loaded. Call `dashboard.screen(...)` or click Load / rebuild for a new universe.


In [ ]:
RUN_LOCAL_EXAMPLE = False

if RUN_LOCAL_EXAMPLE:
    dashboard.set_columns([
        "TICKER", "SIDE", "EXPIRY", "STRIKE", "IV_PCT", "PREMIUM_PCT_STRIKE"
    ])
    dashboard.rank_metric.value = "PREMIUM_PCT_STRIKE"
    dashboard.rank_side.value = "PUT"
    # Same selected columns as the dashboard; numeric percentage points:
    concise_result = dashboard.current_table()
    display(concise_result)

    # Alternative: pure DataFrame API, full columns, no dashboard-control changes.
    full_local_ranking = rank_options(
        dashboard.raw, side="PUT", price_basis="MID",
        rank_by="IV_PCT", iv_unit=dashboard.iv_units.value, top_n=20,
    )


## Data quality and operating notes

**IV source and units.** The supplied adapter requests `bq.data.ivol()` on each selected option ID, with an empty `IVOL_KWARGS` dictionary. It does not assume that the field is the IV implied by your chosen premium basis. The optional fields `IV_RAW`, `IV_SOURCE`, `IV_BASIS`, `IV_OBSERVATION_DATE`, and `IV_INPUT_UNIT` make this explicit. The default treats the native result as percentage points. If changing the field or using another convention, verify its metadata in your licensed BQuant field documentation and set the explicit unit convention accordingly. Do not infer the convention solely from whether values exceed one.

**Unavailable IV.** An IV request failure is recorded in Errors, and `IV_PCT` remains blank. In this case IV ranking may be empty even when price data exist; switch Rank by to one of the percentage-premium metrics to inspect the available premiums. No IV is manufactured, and premium % is never substituted for IV.

**Stock input.** Custom/Single mode does not require an index-membership request. Input IDs are preserved, including qualified markets such as UW or HK. When Bloomberg normalizes a returned stock ID, the adapter queries unresolved inputs individually rather than guessing a multi-row mapping. Duplicate option IDs from alias stock inputs are retained once, under the first input, with a diagnostic. Coverage reports missing stock prices, unavailable chains, empty expiry/strike ranges and request errors.

**Currencies and native units.** Blank currency filter means all. No FX conversion or contract multiplier is applied. Known mismatches between quote and denominator currencies suppress the affected premium ratios; missing metadata is labeled but allowed by default. Use the Advanced option to exclude missing metadata. Raw-premium ranking across multiple known currencies is blocked. Different local quote conventions and adjusted contract deliverables still require checking. Relative-strike bounds assume compatible stock and strike numeric units; use absolute bounds when that is not established. Neither IV ranking nor premium ranking certifies a contract as cheap or expensive relative to fair value.

**Snapshot behavior.** Last, bid, ask and IV may be delayed, stale or unavailable. BQL observation dates are not certified intraday exchange timestamps. Application retrieval time is not quote time. Multi-batch scans are not simultaneous snapshots. MID is a calculated indicative quote, not a trade. The midpoint is left blank for crossed, invalid, currency-mismatched or observation-date-mismatched bid/ask values. The code does not place trades.

**Refresh and query bounds.** Changing custom stocks or strike/expiry bounds requires Load / rebuild. Refresh quotes only refreshes the existing contract IDs and stock observations; it does not discover newly qualifying strikes or expiries. Nearest-target expiries can differ by stock; Exact date uses a common expiry. Same-day expiry eligibility does not certify that trading is still open.

**Results.** `dashboard.ranked` retains full ranked fields, while `dashboard.current_table()` returns selected columns. `dashboard.raw` retains every loaded contract, before ranking filters. Top N is an overall cap, including when Both sides is selected. Highest per stock / side retains one call and one put per stock when available. Search applies within Top N but can match hidden metadata. CSV contains all matching rows and selected columns, not just the current page. Follow your Bloomberg/exchange data-use terms when exporting.

### Verification

Python syntax and notebook structure were validated. The offline regression suite passes **66 tests** using simulated Bloomberg responses and simulated observable widgets. The tests cover custom input mapping, outside-index inputs, IV units and missingness, premium ratios, sorting, quote quality, column selection, views, refresh behavior, and selected-column exports. **No live Bloomberg query, account entitlement, actual ipywidgets backend, or BQuant browser frontend was tested in this environment.**

### Technical references

The BQL IV adapter follows the original author's published `bq.data.ivol()` option-chain query example; that example does not validate current entitlements or the API schema in your environment. Widget selection uses the Jupyter widget interface. Pricing terminology is described by the Options Industry Council. The references are not live data sources for this notebook.

- BQL option-chain / contract-IV example (original implementation): https://medium.com/@annievelcu/visualising-the-effects-of-the-russo-ukrainian-war-in-the-markets-case-study-with-bql-and-bquant-bc908e782206
- Jupyter widget documentation: https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html
- Options Industry Council, Options Pricing: https://www.optionseducation.org/optionsoverview/options-pricing
